# Posterior for local recombination rate using SBI + regression for summary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import Phylo
import torch
from torch.distributions import Uniform
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.inference import NPE_C
from sbi.analysis import plot_summary
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from statsmodels.gam.api import GLMGam, BSplines
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import sys
sys.path.append('../pysimARG')
from clonal_genealogy import ClonalTree
from newick_to_tree import newick_to_tree
from discrete_uniform import DiscreteUniform

torch_device = "cpu"

## Load simulation data

Load genome data and clonal tree.

In [ ]:
genome_simbac = np.loadtxt("../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)

In [ ]:
genome_simbac.shape

In [ ]:
np.random.seed(100)
clonal_tree = ClonalTree(n=30)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
clonal_tree.edge = edge
clonal_tree.node_height = node_height
clonal_tree.height = np.max(node_height)
clonal_tree.length = np.sum(edge[:, 2])

## Summary statistics

### Get summary statistics from SimBac data

In [ ]:
x_500_SB = np.loadtxt("../data/x_500_SB.csv", delimiter=",")
x_2000_SB = np.loadtxt("../data/x_2000_SB.csv", delimiter=",")
x_6000_SB = np.loadtxt("../data/x_6000_SB.csv", delimiter=",")

x_500_SB.shape, x_2000_SB.shape, x_6000_SB.shape

### Load simulations from ClonalOrigin model

In [ ]:
x_500_CO = np.loadtxt("../data/x_500_CO.csv", delimiter=",")
x_2000_CO = np.loadtxt("../data/x_2000_CO.csv", delimiter=",")
x_6000_CO = np.loadtxt("../data/x_6000_CO.csv", delimiter=",")

x_500_CO.shape, x_2000_CO.shape, x_6000_CO.shape

In [ ]:
theta1 = np.loadtxt('../data/theta1.csv', delimiter=",")
x1 = np.loadtxt('../data/x1.csv', delimiter=",")

theta2 = np.loadtxt('../data/theta2.csv', delimiter=",")
x2 = np.loadtxt('../data/x2.csv', delimiter=",")

x = np.vstack([x1, x2])
theta = np.vstack([theta1, theta2])

print(theta.shape, x.shape)

In [ ]:
theta = torch.tensor(theta, device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = torch.tensor(x, device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

## NPE

### Create prior to pass range knowledge to NPE

In [ ]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))

prior = MultipleIndependent(
    dists=[prior_rho, prior_L],
    validate_args=False,
    device=torch_device
)

### Linear regression

Consider the linear regression
$$
y_i = \beta_0 + \sum_{j=1}^n (\beta_{j1} x_{ij} + \beta_{j2} x_{ij}^2 + \beta_{j3} x_{ij}^3 + \beta_{j4} x_{ij}^4) + \epsilon_i,
$$
take $\hat\beta_0 + \sum_{j=1}^n (\hat\beta_{j1} x_j + \hat\beta_{j2} x_j^2 + \hat\beta_{j3} x_j^3 + \hat\beta_{j4} x_j^4)$ as the summary statistics.

In [ ]:
seed = 100
num_posterior_samples=1000
learning_rate = 0.0005

inference1 = NPE_C(prior=prior, density_estimator="nsf", device=torch_device)
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
x_poly = np.hstack([x_numpy**i for i in range(1, 5)])
y = theta_numpy[:, 0:1]

print(x_poly.shape, y.shape)

model = LinearRegression(fit_intercept=True)
model.fit(x_poly, y)

predictions = model.predict(x_poly)
score = model.score(x_poly, y)
print(f"Model R^2 Score: {score}")

In [ ]:
x1_numpy = np.hstack([predictions, theta_numpy[:, 1:2]])
x1 = torch.tensor(x1_numpy, device=torch_device)
x1 = x1.to(torch.float32)

x1.shape, x1.dtype

In [ ]:
density_estimator1 = inference1.append_simulations(theta, x1).train(
    max_num_epochs=500, learning_rate=learning_rate
)
posterior1 = inference1.build_posterior(density_estimator1)

In [ ]:
fig, axes = plot_summary(
    inference1, 
    tags=["training_loss", "validation_loss"], 
    figsize=(8, 4)
)
plt.title("Training and Validation Loss")
plt.show()

#### SimBac observations

In [ ]:
theta1_500 = np.full((100, num_posterior_samples, 2), np.nan)
theta1_2000 = np.full((100, num_posterior_samples, 2), np.nan)
theta1_6000 = np.full((100, num_posterior_samples, 2), np.nan)

theta1_500.shape, theta1_2000.shape, theta1_6000.shape

In [ ]:
target_x_poly = np.hstack([x_500_SB**i for i in range(1, 5)])
pred_obs = model.predict(target_x_poly)
x_obs_numpy = np.hstack([pred_obs, x_500_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)


for i in range(100):
    theta_post = posterior1.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta1_500[i, :, :] = theta_post.cpu().numpy()

In [ ]:
target_x_poly = np.hstack([x_2000_SB**i for i in range(1, 5)])
pred_obs = model.predict(target_x_poly)
x_obs_numpy = np.hstack([pred_obs, x_2000_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior1.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta1_2000[i, :, :] = theta_post.cpu().numpy()

In [ ]:
target_x_poly = np.hstack([x_6000_SB**i for i in range(1, 5)])
pred_obs = model.predict(target_x_poly)
x_obs_numpy = np.hstack([pred_obs, x_6000_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior1.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta1_6000[i, :, :] = theta_post.cpu().numpy()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta1_500[0, :, 0], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta1_2000[0, :, 0], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta1_6000[0, :, 0], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta1_500[i, :, 0], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta1_2000[i, :, 0], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta1_6000[i, :, 0], color='orange', linewidth=0.3, alpha=0.2)
plt.axvline(x=0.02, color='red', linestyle='dashed', label='True value')
plt.xlabel(r'$\rho_s$')
plt.xlim(0.0, 0.1)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta1_500[0, :, 1], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta1_2000[0, :, 1], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta1_6000[0, :, 1], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta1_500[i, :, 1], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta1_2000[i, :, 1], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta1_6000[i, :, 1], color='orange', linewidth=0.3, alpha=0.2)
plt.xlabel('N')
plt.xlim(100.0, 10000.0)
plt.legend()
plt.show()

In [ ]:
df_500 = pd.DataFrame(theta1_500[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_500)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 500)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_2000 = pd.DataFrame(theta1_2000[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_2000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 2000)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_6000 = pd.DataFrame(theta1_6000[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_6000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 6000)', fontsize=12, fontweight='bold')
plt.show()

#### ClonalOrigin observations

In [ ]:
theta1_500_CO = np.full((100, num_posterior_samples, 2), np.nan)
theta1_2000_CO = np.full((100, num_posterior_samples, 2), np.nan)
theta1_6000_CO = np.full((100, num_posterior_samples, 2), np.nan)

theta1_500_CO.shape, theta1_2000_CO.shape, theta1_6000_CO.shape

In [ ]:
target_x_poly = np.hstack([x_500_CO**i for i in range(1, 5)])
pred_obs = model.predict(target_x_poly)
x_obs_numpy = np.hstack([pred_obs, x_500_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior1.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta1_500_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
target_x_poly = np.hstack([x_2000_CO**i for i in range(1, 5)])
pred_obs = model.predict(target_x_poly)
x_obs_numpy = np.hstack([pred_obs, x_2000_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior1.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta1_2000_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
target_x_poly = np.hstack([x_6000_CO**i for i in range(1, 5)])
pred_obs = model.predict(target_x_poly)
x_obs_numpy = np.hstack([pred_obs, x_6000_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior1.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta1_6000_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta1_500_CO[0, :, 0], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta1_2000_CO[0, :, 0], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta1_6000_CO[0, :, 0], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta1_500_CO[i, :, 0], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta1_2000_CO[i, :, 0], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta1_6000_CO[i, :, 0], color='orange', linewidth=0.3, alpha=0.2)
plt.axvline(x=0.02, color='red', linestyle='dashed', label='True value')
plt.xlabel(r'$\rho_s$')
plt.xlim(0.0, 0.1)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta1_500_CO[0, :, 1], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta1_2000_CO[0, :, 1], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta1_6000_CO[0, :, 1], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta1_500_CO[i, :, 1], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta1_2000_CO[i, :, 1], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta1_6000_CO[i, :, 1], color='orange', linewidth=0.3, alpha=0.2)
plt.xlabel('N')
plt.xlim(100.0, 10000.0)
plt.legend()
plt.show()

In [ ]:
df_500 = pd.DataFrame(theta1_500_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_500)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 500)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_2000 = pd.DataFrame(theta1_2000_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_2000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 2000)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_6000 = pd.DataFrame(theta1_6000_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_6000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 6000)', fontsize=12, fontweight='bold')
plt.show()

### Generalized Additive Models (GAM)

Consider the GAM equation
$$
y_i = \beta_0 + \sum_{j=1}^n f_j(x_{ij}) + \epsilon_i,
$$
take $\hat\beta_0 + \sum_{j=1}^n f_j(x_j)$ as the summary statistics.

In [ ]:
seed = 100
num_posterior_samples=1000
learning_rate = 0.0005

inference2 = NPE_C(prior=prior, density_estimator="nsf", device=torch_device)
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
degrees_of_freedom = [5] * 46 
polynomial_degrees = [3] * 46

spline_basis = BSplines(x_numpy, df=degrees_of_freedom, degree=polynomial_degrees)
exog_linear = pd.DataFrame({'const': np.ones(x_numpy.shape[0])})

penalty_weights = np.array([1.0] * 46)
gam_model = GLMGam(endog=theta_numpy[:, 0:1], exog=exog_linear, smoother=spline_basis, alpha=penalty_weights)

gam_results = gam_model.fit()

print(gam_results.summary())

In [ ]:
pred_gam = gam_results.predict()
x2_numpy = np.hstack([pred_gam.reshape(-1, 1), theta_numpy[:, 1:2]])
x2 = torch.tensor(x2_numpy, device=torch_device)
x2 = x2.to(torch.float32)

x2.shape, x2.dtype

In [ ]:
density_estimator2 = inference2.append_simulations(theta, x2).train(
    max_num_epochs=500, learning_rate=learning_rate
)
posterior2 = inference2.build_posterior(density_estimator2)

In [ ]:
fig, axes = plot_summary(
    inference2, 
    tags=["training_loss", "validation_loss"], 
    figsize=(8, 4)
)
plt.title("Training and Validation Loss")
plt.show()

#### SimBac observations

In [ ]:
theta2_500 = np.full((100, num_posterior_samples, 2), np.nan)
theta2_2000 = np.full((100, num_posterior_samples, 2), np.nan)
theta2_6000 = np.full((100, num_posterior_samples, 2), np.nan)

theta2_500.shape, theta2_2000.shape, theta2_6000.shape

In [ ]:
new_exog_linear = pd.DataFrame({'const': np.ones(x_500_SB.shape[0])})
pred_obs = gam_results.predict(exog=new_exog_linear, exog_smooth=x_500_SB)
x_obs_numpy = np.hstack([pred_obs.reshape(-1, 1), x_500_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior2.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta2_500[i, :, :] = theta_post.cpu().numpy()

In [ ]:
new_exog_linear = pd.DataFrame({'const': np.ones(x_2000_SB.shape[0])})
pred_obs = gam_results.predict(exog=new_exog_linear, exog_smooth=x_2000_SB)
x_obs_numpy = np.hstack([pred_obs.reshape(-1, 1), x_2000_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior2.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta2_2000[i, :, :] = theta_post.cpu().numpy()

In [ ]:
new_exog_linear = pd.DataFrame({'const': np.ones(x_6000_SB.shape[0])})
pred_obs = gam_results.predict(exog=new_exog_linear, exog_smooth=x_6000_SB)
x_obs_numpy = np.hstack([pred_obs.reshape(-1, 1), x_6000_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior2.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta2_6000[i, :, :] = theta_post.cpu().numpy()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
# sns.kdeplot(theta2_500[0, :, 0], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta2_2000[0, :, 0], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta2_6000[0, :, 0], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    # sns.kdeplot(theta2_500[i, :, 0], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta2_2000[i, :, 0], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta2_6000[i, :, 0], color='orange', linewidth=0.3, alpha=0.2)
plt.axvline(x=0.02, color='red', linestyle='dashed', label='True value')
plt.xlabel(r'$\rho_s$')
plt.xlim(0.0, 0.1)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
# sns.kdeplot(theta2_500[0, :, 1], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta2_2000[0, :, 1], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta2_6000[0, :, 1], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    # sns.kdeplot(theta2_500[i, :, 1], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta2_2000[i, :, 1], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta2_6000[i, :, 1], color='orange', linewidth=0.3, alpha=0.2)
plt.xlabel('N')
plt.xlim(100.0, 10000.0)
plt.legend()
plt.show()

In [ ]:
df_500 = pd.DataFrame(theta2_500[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_500)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 500)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_2000 = pd.DataFrame(theta2_2000[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_2000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 2000)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_6000 = pd.DataFrame(theta2_6000[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_6000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 6000)', fontsize=12, fontweight='bold')
plt.show()

#### ClonalOrigin observations

In [ ]:
theta2_500_CO = np.full((100, num_posterior_samples, 2), np.nan)
theta2_2000_CO = np.full((100, num_posterior_samples, 2), np.nan)
theta2_6000_CO = np.full((100, num_posterior_samples, 2), np.nan)

theta2_500_CO.shape, theta2_2000_CO.shape, theta2_6000_CO.shape

In [ ]:
new_exog_linear = pd.DataFrame({'const': np.ones(x_500_CO.shape[0])})
pred_obs = gam_results.predict(exog=new_exog_linear, exog_smooth=x_500_CO)
x_obs_numpy = np.hstack([pred_obs.reshape(-1, 1), x_500_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior2.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta2_500_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
new_exog_linear = pd.DataFrame({'const': np.ones(x_2000_CO.shape[0])})
pred_obs = gam_results.predict(exog=new_exog_linear, exog_smooth=x_2000_CO)
x_obs_numpy = np.hstack([pred_obs.reshape(-1, 1), x_2000_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior2.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta2_2000_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
new_exog_linear = pd.DataFrame({'const': np.ones(x_6000_CO.shape[0])})
pred_obs = gam_results.predict(exog=new_exog_linear, exog_smooth=x_6000_CO)
x_obs_numpy = np.hstack([pred_obs.reshape(-1, 1), x_6000_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior2.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta2_6000_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta2_500_CO[0, :, 0], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta2_2000_CO[0, :, 0], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta2_6000_CO[0, :, 0], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta2_500_CO[i, :, 0], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta2_2000_CO[i, :, 0], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta2_6000_CO[i, :, 0], color='orange', linewidth=0.3, alpha=0.2)
plt.axvline(x=0.02, color='red', linestyle='dashed', label='True value')
plt.xlabel(r'$\rho_s$')
plt.xlim(0.0, 0.1)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta2_500_CO[0, :, 1], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta2_2000_CO[0, :, 1], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta2_6000_CO[0, :, 1], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta2_500_CO[i, :, 1], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta2_2000_CO[i, :, 1], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta2_6000_CO[i, :, 1], color='orange', linewidth=0.3, alpha=0.2)
plt.xlabel('N')
plt.xlim(100.0, 10000.0)
plt.legend()
plt.show()

In [ ]:
df_500 = pd.DataFrame(theta2_500_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_500)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 500)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_2000 = pd.DataFrame(theta2_2000_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_2000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 2000)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_6000 = pd.DataFrame(theta2_6000_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_6000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 6000)', fontsize=12, fontweight='bold')
plt.show()

### Neural Network

Use a neural network to fit `X` and `theta`. Then use the predicted values as summary statistics.

In [ ]:
seed = 100
num_posterior_samples=1000
learning_rate = 0.0005

inference3 = NPE_C(prior=prior, density_estimator="nsf", device=torch_device)
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_numpy)

model = Sequential([                                 # input layer - 46 features
    Dense(64, activation='relu', input_shape=(46,)), # layer 1 - 64 neurons
    Dense(32, activation='relu'),                    # layer 2 - 32 neurons
    Dense(1)                                         # output layer - 1 neuron
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

history = model.fit(
    x_scaled, 
    theta_numpy[:, 0:1],
    epochs=80, 
    batch_size=32, 
    validation_split=0.2,
    verbose=1 # Shows a progress bar
)

In [ ]:
pred_nn = model.predict(x_scaled)
x3_numpy = np.hstack([pred_nn, theta_numpy[:, 1:2]])
x3 = torch.tensor(x3_numpy, device=torch_device)
x3 = x3.to(torch.float32)

x3.shape, x3.dtype

In [ ]:
density_estimator3 = inference3.append_simulations(theta, x3).train(
    max_num_epochs=500, learning_rate=learning_rate
)
posterior3 = inference3.build_posterior(density_estimator3)

In [ ]:
fig, axes = plot_summary(
    inference3, 
    tags=["training_loss", "validation_loss"], 
    figsize=(8, 4)
)
plt.title("Training and Validation Loss")
plt.show()

#### SimBac observations

In [ ]:
theta3_500 = np.full((100, num_posterior_samples, 2), np.nan)
theta3_2000 = np.full((100, num_posterior_samples, 2), np.nan)
theta3_6000 = np.full((100, num_posterior_samples, 2), np.nan)

theta3_500.shape, theta3_2000.shape, theta3_6000.shape

In [ ]:
x_new_scaled = scaler.transform(x_500_SB)
pred_obs = model.predict(x_new_scaled)
x_obs_numpy = np.hstack([pred_obs, x_500_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior3.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta3_500[i, :, :] = theta_post.cpu().numpy()

In [ ]:
x_new_scaled = scaler.transform(x_2000_SB)
pred_obs = model.predict(x_new_scaled)
x_obs_numpy = np.hstack([pred_obs, x_2000_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior3.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta3_2000[i, :, :] = theta_post.cpu().numpy()

In [ ]:
x_new_scaled = scaler.transform(x_6000_SB)
pred_obs = model.predict(x_new_scaled)
x_obs_numpy = np.hstack([pred_obs, x_6000_SB[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior3.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta3_6000[i, :, :] = theta_post.cpu().numpy()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta3_500[0, :, 0], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta3_2000[0, :, 0], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta3_6000[0, :, 0], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta3_500[i, :, 0], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta3_2000[i, :, 0], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta3_6000[i, :, 0], color='orange', linewidth=0.3, alpha=0.2)
plt.axvline(x=0.02, color='red', linestyle='dashed', label='True value')
plt.xlabel(r'$\rho_s$')
plt.xlim(0.0, 0.1)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta3_500[0, :, 1], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta3_2000[0, :, 1], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta3_6000[0, :, 1], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta3_500[i, :, 1], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta3_2000[i, :, 1], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta3_6000[i, :, 1], color='orange', linewidth=0.3, alpha=0.2)
plt.xlabel('N')
plt.xlim(100.0, 10000.0)
plt.legend()
plt.show()

In [ ]:
df_500 = pd.DataFrame(theta3_500[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_500)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 500)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_2000 = pd.DataFrame(theta3_2000[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_2000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 2000)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_6000 = pd.DataFrame(theta3_6000[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_6000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 6000)', fontsize=12, fontweight='bold')
plt.show()

#### ClonalOrigin observations

In [ ]:
theta3_500_CO = np.full((100, num_posterior_samples, 2), np.nan)
theta3_2000_CO = np.full((100, num_posterior_samples, 2), np.nan)
theta3_6000_CO = np.full((100, num_posterior_samples, 2), np.nan)

theta3_500_CO.shape, theta3_2000_CO.shape, theta3_6000_CO.shape

In [ ]:
x_new_scaled = scaler.transform(x_500_CO)
pred_obs = model.predict(x_new_scaled)
x_obs_numpy = np.hstack([pred_obs, x_500_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior3.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta3_500_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
x_new_scaled = scaler.transform(x_2000_CO)
pred_obs = model.predict(x_new_scaled)
x_obs_numpy = np.hstack([pred_obs, x_2000_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior3.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta3_2000_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
x_new_scaled = scaler.transform(x_6000_CO)
pred_obs = model.predict(x_new_scaled)
x_obs_numpy = np.hstack([pred_obs, x_6000_CO[:, -1].reshape(-1, 1)])
x_obs = torch.tensor(x_obs_numpy, device=torch_device)
x_obs = x_obs.to(torch.float32)

for i in range(100):
    theta_post = posterior3.sample((num_posterior_samples,), x=x_obs[i, :], show_progress_bars=False)
    theta3_6000_CO[i, :, :] = theta_post.cpu().numpy()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta3_500_CO[0, :, 0], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta3_2000_CO[0, :, 0], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta3_6000_CO[0, :, 0], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta3_500_CO[i, :, 0], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta3_2000_CO[i, :, 0], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta3_6000_CO[i, :, 0], color='orange', linewidth=0.3, alpha=0.2)
plt.axvline(x=0.02, color='red', linestyle='dashed', label='True value')
plt.xlabel(r'$\rho_s$')
plt.xlim(0.0, 0.1)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_style('whitegrid')
sns.kdeplot(theta3_500_CO[0, :, 1], label='N = 500', color='blue', linewidth=2, alpha=1.0)
sns.kdeplot(theta3_2000_CO[0, :, 1], label='N = 2000', color='green', linewidth=2, alpha=1.0)
sns.kdeplot(theta3_6000_CO[0, :, 1], label='N = 6000', color='orange', linewidth=2, alpha=1.0)
for i in range(1, 100):
    sns.kdeplot(theta3_500_CO[i, :, 1], color='blue', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta3_2000_CO[i, :, 1], color='green', linewidth=0.3, alpha=0.2)
    sns.kdeplot(theta3_6000_CO[i, :, 1], color='orange', linewidth=0.3, alpha=0.2)
plt.xlabel('N')
plt.xlim(100.0, 10000.0)
plt.legend()
plt.show()

In [ ]:
df_500 = pd.DataFrame(theta3_500_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_500)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 500)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_2000 = pd.DataFrame(theta3_2000_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_2000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 2000)', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
df_6000 = pd.DataFrame(theta3_6000_CO[0, :, :], columns=[r"$\rho_s$", "N"])
sns.pairplot(df_6000)
plt.suptitle('Pair Plot of NPE-C Posterior Samples (N = 6000)', fontsize=12, fontweight='bold')
plt.show()